# Bài Thực Hành Lab 02: Tóm Tắt Dữ Liệu Phân Loại & Dữ Liệu Số
**Học phần:** UET.MAT1052 — Xác suất thống kê (VNU-UET)
---
### Mục tiêu bài học:
1. **Dữ liệu phân loại:** Tần số, tỉ lệ, bảng liên hợp `crosstab()`, hiểu rõ các mẫu số `normalize='index'` / `'columns'` / `'all'`.
2. **Dữ liệu số:** Trung bình (Mean), Trung vị (Median), Phương sai, Độ lệch chuẩn, Tứ phân vị, IQR, `describe()`.
3. **Kiểm chứng lý thuyết:** Tính bền vững (Robustness) trước ngoại lai; Biến đổi tuyến tính $Y = aX + b$; Tổn thất bình phương $L(a) = \sum (x_i - a)^2$.
4. **Trực quan hóa:** Histogram và Boxplot phát hiện ngoại lai với Matplotlib và Seaborn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Thiết lập giao diện hiển thị
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 4.5)

## 1. Tóm Tắt Dữ Liệu Phân Loại (Palmer Penguins)
Khảo sát biến loài (`species`) và đảo sinh sống (`island`).

In [ ]:
penguins = pd.read_csv("du-lieu/penguins.csv")

# 1. Số đếm và tỉ lệ một chiều
print("--- Số đếm từng loài ---")
display(penguins['species'].value_counts())

print("\n--- Tỉ lệ từng loài ---")
display(penguins['species'].value_counts(normalize=True).round(3))

# 2. Bảng liên hợp hai chiều (Contingency table)
print("\n--- Bảng liên hợp species x island (có tổng viền) ---")
display(pd.crosstab(penguins['species'], penguins['island'], margins=True))

### Thử thách về mẫu số trong `crosstab` (W02-PY01)
- `normalize='index'`: Tính theo hàng (tổng mỗi hàng = 1.0) -> *Phân bố các đảo trong từng loài*.
- `normalize='columns'`: Tính theo cột (tổng mỗi cột = 1.0) -> *Cơ cấu các loài trên từng đảo*.

In [ ]:
print("--- Tỉ lệ theo hàng (Đảo trong từng loài) ---")
display(pd.crosstab(penguins['species'], penguins['island'], normalize='index').round(3))

print("\n--- Tỉ lệ theo cột (Loài trên từng đảo) ---")
display(pd.crosstab(penguins['species'], penguins['island'], normalize='columns').round(3))

## 2. Tóm Tắt Dữ Liệu Số & Kiểm Chứng Lý Thuyết

In [ ]:
# Dãy gốc STAT 20
x = pd.Series([8, 11, 7, 7, 8, 11, 9, 6, 10, 7, 9])
print(f"Dãy gốc: Mean = {x.mean():.3f}, Median = {x.median()}, SD = {x.std():.3f}, IQR = {x.quantile(0.75)-x.quantile(0.25):.3f}")

# Thay 6 bằng -200 (W02-EX07 / W02-CH02)
x_bad = pd.Series([8, 11, 7, 7, 8, 11, 9, -200, 10, 7, 9])
print(f"Sau thay đổi: Mean = {x_bad.mean():.3f}, Median = {x_bad.median()}, SD = {x_bad.std():.3f}, IQR = {x_bad.quantile(0.75)-x_bad.quantile(0.25):.3f}")

# Nhận xét: Mean và SD thay đổi đột ngột (không bền vững), trong khi Median và IQR không đổi (bền vững)
# Kiểm chứng tổn thất bình phương L(a) = sum((x_i - a)^2) đạt cực tiểu tại a = x.mean()
mean_val = x.mean()
loss_mean = ((x - mean_val) ** 2).sum()
print(f"\nKiểm chứng hàm tổn thất bình phương L(a) = sum((x_i - a)^2):")
print(f"- L(a = Mean = {mean_val:.3f}) = {loss_mean:.3f}")
for a_test in [mean_val - 1, mean_val + 1, float(x.median())]:
    print(f"- L(a = {a_test:.3f}) = {((x - a_test) ** 2).sum():.3f} (luôn lớn hơn tại Mean)")

### Kiểm chứng biến đổi thang đo tuyến tính $Y = 3X + 5$ (W02-PY05)

In [ ]:
y = 3 * x + 5
print("X: Mean =", round(x.mean(), 3), "SD =", round(x.std(), 3), "IQR =", x.quantile(0.75) - x.quantile(0.25))
print("Y: Mean =", round(y.mean(), 3), "SD =", round(y.std(), 3), "IQR =", y.quantile(0.75) - y.quantile(0.25))

# Kiểm chứng công thức biến đổi thang đo tuyến tính: Y = aX + b (với a=3, b=5)
# Mean(Y) = a * Mean(X) + b
# SD(Y)   = |a| * SD(X)
# IQR(Y)  = |a| * IQR(X)
print("\n--- So sánh với công thức lý thuyết (a=3, b=5) ---")
print(f"3 * Mean(X) + 5 = {3 * x.mean() + 5:.3f} == Mean(Y) ({y.mean():.3f})")
print(f"3 * SD(X)       = {3 * x.std():.3f} == SD(Y) ({y.std():.3f})")
print(f"3 * IQR(X)      = {3 * (x.quantile(0.75) - x.quantile(0.25)):.3f} == IQR(Y) ({y.quantile(0.75) - y.quantile(0.25):.3f})")

## 3. Trực Quan Hóa Với Histogram & Boxplot

In [ ]:
# 1. Histogram khối lượng cơ thể chim cánh cụt
plt.figure(figsize=(8, 4.5))
sns.histplot(data=penguins, x="body_mass_g", bins=20, kde=True, color="#0284c7")
plt.title("Phân Phối Khối Lượng Chim Cánh Cụt Palmer (g)", fontsize=13, fontweight='bold')
plt.xlabel("Khối lượng cơ thể (g)")
plt.ylabel("Tần số")
plt.show()

# 2. Boxplot so sánh khối lượng giữa 3 loài chim cánh cụt
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=penguins, x="species", y="body_mass_g", hue="species", palette=["#38bdf8", "#ff7a59", "#10b981"], legend=False)
plt.title("So Sánh Khối Lượng Giữa Các Loài Chim Cánh Cụt", fontsize=13, fontweight='bold')
plt.xlabel("Loài chim")
plt.ylabel("Khối lượng (g)")
plt.show()

## 4. Phân Tích Khảo Sát Sinh Viên UET (`uet_class_survey.csv`)

In [ ]:
survey = pd.read_csv("du-lieu/uet_class_survey.csv")
display(survey.head())

print("--- Thống kê mô tả điểm GPA ---")
display(survey['gpa'].describe().round(3))

# Boxplot GPA theo ngành học
plt.figure(figsize=(9, 5))
sns.boxplot(data=survey, x="nganh_hoc", y="gpa", hue="nganh_hoc", palette="Set2", legend=False)
plt.title("So Sánh Phân Bố GPA Giữa Các Ngành Đào Tạo UET", fontsize=13, fontweight='bold')
plt.xlabel("Ngành Học")
plt.ylabel("Điểm GPA")
plt.show()